# Week 8 Assignment - Single Agent Pipeline

**Name:** Parth Moholkar
**Domain:** Data Science Internship, Celebal Technologies

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries -> Calculator Tool
- Keyword extraction -> Keyword Tool
- General queries -> Direct response

## Tool 1 - Calculator

In [1]:
import multiprocessing

def _evaluate_expression(expression, output_queue):
    try:
        output_queue.put(str(eval(expression)))
    except Exception:
        output_queue.put("Error in calculation")

def calculator(expression: str) -> str:
    output_queue = multiprocessing.Queue()
    process = multiprocessing.Process(target=_evaluate_expression, args=(expression, output_queue))
    process.start()
    process.join(timeout=2)

    if process.is_alive():
        process.terminate()
        process.join()
        return "Error in calculation (expression too complex)"

    if not output_queue.empty():
        return output_queue.get()
    return "Error in calculation"

## Tool 2 - Keyword Extractor

In [2]:
STOPWORDS = {"about", "which", "there", "their", "where", "would",
             "could", "should", "these", "those", "being", "while", "after", "before"}

def extract_keywords(text: str) -> list:
    try:
        words = (w.lower().strip(".,!?;:") for w in text.split())
        keywords = list(dict.fromkeys(w for w in words if len(w) > 4 and w not in STOPWORDS))
        return keywords[:5]
    except Exception:
        return []

## Tool 3 - Word Counter (Bonus)

In [3]:
def count_words(text: str) -> dict:
    try:
        return {"words": len(text.split()), "characters": len(text)}
    except Exception:
        return {"words": 0, "characters": 0}

## Logging Setup (Bonus)

In [4]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger("agent")

## Agent Logic

In [5]:
import re

CALCULATION_TRIGGERS = ["calculate", "compute", "solve"]
KEYWORD_TRIGGERS = ["keywords", "keyword"]
COUNT_TRIGGERS = ["count", "length"]

def is_bare_expression(query: str) -> bool:
    return bool(re.fullmatch(r"[0-9+\-*/().\s^]+", query.strip()))

def agent(query: str) -> dict:
    query_lower = query.lower()

    try:
        if any(trigger in query_lower for trigger in CALCULATION_TRIGGERS) or is_bare_expression(query):
            expression = re.sub(r"[^0-9+\-*/().\s^]", "", query).replace("^", "**")
            result = calculator(expression.strip())
            response_type = "error" if result.startswith("Error") else "calculation"
            response = {"type": response_type, "result": result}

        elif any(trigger in query_lower for trigger in KEYWORD_TRIGGERS):
            response = {"type": "keywords", "result": extract_keywords(query)}

        elif any(trigger in query_lower for trigger in COUNT_TRIGGERS):
            response = {"type": "count", "result": count_words(query)}

        else:
            response = {"type": "general", "result": f"No specific tool matched - treating as a general query: {query}"}

    except Exception as e:
        response = {"type": "error", "result": str(e)}

    logger.info(f"Query: {query!r} -> {response}")
    return response

## Expected Output Format

```
{
  "type": "calculation / keywords / count / general / error",
  "result": ...
}
```

## Test Cases

In [6]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Count the length of this sentence",
    "Calculate 10 / 0",
    "Calculate 9**9**9**9"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'artificial', 'intelligence', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'No specific tool matched - treating as a general query: What is machine learning?'}
--------------------------------------------------
Query: Count the length of this sentence
Response: {'type': 'count', 'result': {'words': 6, 'characters': 33}}
--------------------------------------------------
Query: Calculate 10 / 0
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: Calculate 9**9**9**9
Response: {'type': 'error', 'result': 'Error in calculation (expression too complex)'}
---------

## Interactive Mode

In [ ]:
MAX_TURNS = 10

for turn in range(MAX_TURNS):
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))
else:
    print(f"Reached the {MAX_TURNS}-turn demo limit.")

Enter query (type 'exit' to stop): 20 + 9
Response: {'type': 'calculation', 'result': '29'}
